In [1]:
# 공공데이터 약국 활용 예제 : 건강보험심사평가원_약국정보서비스

In [2]:
import requests
from bs4 import BeautifulSoup

service_key = 'cfd43b74c58abf296f28bf20ee869b1c2df2b0c9dd34551fe8fecb7d69e12c8b'

def getParmacy(service_key : str, page_no : int = 1):
    """공공 데이터에서 약국 정보를 가져와서 딕셔너리로 반환하는 함수"""
    # End Point + 활용신청 상세기능정보 > 상세기능에 있는 주소
    url = "https://apis.data.go.kr/B551182/pharmacyInfoService/getParmacyBasisList"
    params={
        'ServiceKey' : service_key,
        'pageNo' : page_no
    }
    
    try:
        reponse = requests.get(url, params=params)
    
        reponse.raise_for_status()
        # 응답 결과가 xml 파일이어서 변수명이 res_xml로 함(강사 마음)
        res_xml = reponse.text
    
        soup = BeautifulSoup(res_xml, 'lxml-xml')
    
        # # 전체 약국수
        # totalCount = soup.find('totalCount').text
        # print(totalCount)
        
        # 약국들 정보
        items = soup.find_all('item')

        # 받아온 데이터들을 가져오기 위한 리스트 선언
        yadmNms = []
        clCdNms = []
        sidoCdNms = []
        sgguCdNms = []
        addrs = []

        # 받아온 데이터들을 가져와서 리스트에 추가
        for item in items:
            yadmNms.append(item.find('yadmNm').text if item.find('yadmNm').text else None) # 병원명
            clCdNms.append(item.find('clCdNm').text) # 종별 코드명(약국)
            sidoCdNms.append(item.find('sidoCdNm').text) # 시도명
            sgguCdNms.append(item.find('sgguCdNm').text) # 시군구명
            addrs.append(item.find('addr').text) # 주소

        # 받아온 데이터를 딕셔너리로 변환
        rec_dic = {
            '병원명' : yadmNms,
            '종별코드명' : clCdNms,
            '시도명' : sidoCdNms,
            '시군구명' : sgguCdNms,
            '주소' : addrs
        }
        # 최종 데이터가 있는 딕셔너리를 반환
        return rec_dic
    except Exception as e:
        print(f"예외 발생 {e}"),
        return []

In [3]:
import pandas as pd
import time

df = pd.DataFrame({
    '병원명' : [],
    '종별코드명' : [],
    '시도명' : [],
    '시군구명' : [],
    '주소' : []
})
print(df)

Empty DataFrame
Columns: [병원명, 종별코드명, 시도명, 시군구명, 주소]
Index: []


In [4]:
for i in range(1,6):
    tmp_df = pd.DataFrame(getParmacy(service_key, i))
    df = pd.concat([df, tmp_df])
    print(f"{i} 페이지가 로딩되었습니다.")
    time.sleep(i)

1 페이지가 로딩되었습니다.
2 페이지가 로딩되었습니다.
3 페이지가 로딩되었습니다.
4 페이지가 로딩되었습니다.
5 페이지가 로딩되었습니다.


In [5]:
# 인덱스 번호 초기화
df = df.reset_index(drop=True)

In [6]:
# csv 파일로 저장
df.to_csv('약국 목록.csv', index=False, encoding='utf-8-sig')

In [7]:
# csv 파일 읽어오기
df2 = pd.read_csv('약국 목록.csv', encoding='utf-8')

In [8]:
# 경기에 있는 약국 수를 조회
print(f"경기에 있는 약국 수 : {len(df.loc[df['시도명'] == '경기'])}")

경기에 있는 약국 수 : 6


In [9]:
# 결측치가 있는지 확인 : None, NaN, NaT
df.replace("", None)
print(df.isnull().sum())

병원명      0
종별코드명    0
시도명      0
시군구명     0
주소       0
dtype: int64
